# 08 Lab — The Strategy Selection Matrix in Action

Five market setups. For each, we read **direction x IV level x horizon**, build the 2-3 candidate
structures the matrix suggests from the appropriate sample chain, compare them with
`analyzer.summarize` + `viz.plot_compare`, and reason to a pick in markdown.

Chains: `DEMO` (spot 100, IV~25%), `LOWVOL` (spot 185, IV~14%), `HIGHVOL` (spot 62, IV~55%).
IV rank is stated per setup (the samples are single snapshots; treat the stated rank as given).

Runs offline, top-to-bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, viz, data

demo = data.load_sample_chain("DEMO")
lowvol = data.load_sample_chain("LOWVOL")
highvol = data.load_sample_chain("HIGHVOL")

def brief(pos, spot, vol):
    s = analyzer.summarize(pos, spot, vol)
    print(f"{s['label']:<34} net={s['net_premium']:+8.0f} "
          f"maxP={s['max_profit']:>8} maxL={s['max_loss']:>8} POP={s['probability_of_profit']:.2f}")

## Setup 1 — Neutral, HIGH IV, 30 days (HIGHVOL, spot 62)

IV rank ~75. No directional lean, expect the name to stay roughly in a range. High IV → **sell
premium**. Matrix neutral/high-IV cell: iron condor, iron butterfly, short strangle. Candidates
below (30 DTE).

In [ ]:
ic = strategies.iron_condor((50,0.54),(55,1.37),(70,1.32),(72.5,0.87), expiry=30/365)
ib = strategies.iron_butterfly((55,1.37),(62.5,4.14),(62.5,3.85),(70,1.32), expiry=30/365)
strangle = strategies.short_strangle((55,1.37),(70,1.32), expiry=30/365)
for p in (ic, ib, strangle):
    brief(p, 62, 0.56)

In [ ]:
ax = viz.plot_compare([ic, ib, strangle])
ax.set_title("Setup 1: condor vs butterfly vs short strangle (HIGHVOL)")

**Reasoning.** The short strangle has the highest credit and POP but **undefined risk** and
big buying-power use — a tie-breaker veto unless the account can carry it and you tolerate assignment
on either side. The iron butterfly earns the most *defined* credit but has a narrow profit tent
(pin bet). The **iron condor** is the balanced pick: defined risk, a wide profit zone matching a
"stay in the range" view, ~16-delta-ish shorts. **Pick: iron condor**, manage at 50% / 21 DTE.

## Setup 2 — Bullish, LOW IV, 60 days (LOWVOL, spot 185)

IV rank ~15. Moderately bullish, patient. Low IV → **buy premium / debit**. Matrix bullish/low-IV:
long call, bull **call** debit spread, call diagonal / PMCC. Candidates use the 45 DTE chain (proxy
for the horizon) plus a 90 DTE leg for the diagonal.

In [ ]:
long_c = strategies.long_call((185, 4.4), expiry=45/365)
bull_call = strategies.bull_call_spread((185, 4.4), (195, 0.95), expiry=45/365)
diag = strategies.diagonal_spread("call", short=(195, 0.95), long=(180, 9.45),
                                  short_expiry=45/365, long_expiry=90/365)
for p in (long_c, bull_call, diag):
    brief(p, 185, 0.15)

In [ ]:
ax = viz.plot_compare([long_c, bull_call, diag])
ax.set_title("Setup 2: long call vs bull call spread vs call diagonal (LOWVOL)")

**Reasoning.** The naked long call has the most upside but pays full theta and full (cheap,
so acceptable) vega — fine in low IV if you want convexity. The **bull call spread** caps upside at
195 but roughly halves the debit and defines risk — the cleanest expression of "moderately bullish,
patient" with a known max loss. The diagonal adds a long-vega tilt and lets you roll the short leg.
**Pick: bull call spread** for defined risk; upgrade to the diagonal if you want to sell the front
repeatedly. In low IV, we are net *buyers* — correct side of the vol trade.

## Setup 3 — Bullish, HIGH IV, 30 days, willing to own (HIGHVOL, spot 62)

IV rank ~75. Bullish but high IV → **sell premium** on the bull side. Matrix: cash-secured put, bull
**put** credit spread, jade lizard. You would happily own HIGHVOL at 55.

In [ ]:
csp = strategies.cash_secured_put((55, 1.37), expiry=30/365)
bull_put = strategies.bull_put_spread((57.5, 2.06), (52.5, 0.88), expiry=30/365)
jade = strategies.jade_lizard((55, 1.37), (67.5, 1.94), (70, 1.32), expiry=30/365)
for p in (csp, bull_put, jade):
    brief(p, 62, 0.56)
print("jade credit/sh:", round(-jade.net_premium()/100, 2), "call-spread width:", 2.5)

**Reasoning.** All three sell the rich IV. The cash-secured put has the largest downside
commitment (undefined-ish to zero) but you *want* the shares at 55. The bull put spread defines risk.
The **jade lizard** adds a call spread above: check credit/share vs the 2.5 call-spread width — if
credit >= width there is **no upside risk**. Here credit ~ (1.37+1.94-1.32)=1.99 > 2.5? No (1.99 <
2.5), so slight upside risk remains — tighten or collect more. **Pick: jade lizard if you can satisfy
credit >= width; otherwise the bull put spread** for clean defined risk.

## Setup 4 — Neutral PIN, moderate IV, 45 days (DEMO, spot 100)

IV rank ~40 (middling). Expect DEMO to pin near 100. Neutral cell, not clearly high or low IV: a
**calendar** at the pin (long back-month vega) vs a cheap defined **long butterfly**.

In [ ]:
cal = strategies.calendar_spread("call", 100, front_expiry=21/365, front_premium=2.66,
                                 back_expiry=45/365, back_premium=3.91)
fly = strategies.long_call_butterfly((95, 7.05), (100, 3.91), (105, 1.85), expiry=45/365)
for p in (cal, fly):
    brief(p, 100, 0.26)

A calendar is mixed-expiry — its true tent needs `pnl_at` at front expiry, so compare the
fly's expiry payoff against the calendar's front-expiry curve rather than a naive overlay.

In [ ]:
spots = np.linspace(88, 112, 121)
from optionslab import payoff
fig, ax = plt.subplots()
ax.plot(spots, payoff.pnl_curve(fly, spots), label="long fly (expiry)")
ax.plot(spots, payoff.pnl_curve(cal, spots, t_elapsed=21/365, vol=0.26), label="calendar (front exp)")
ax.axhline(0, color="k", lw=.7); ax.legend(); ax.set_title("Setup 4: fly vs calendar around the pin")

**Reasoning.** Both profit if DEMO pins 100. The **long fly** is a pure defined-risk pin bet
with no vega view. The **calendar** adds long vega — better if you also think IV will hold or rise,
worse if a vol crush is coming. **Pick: fly if you have no vol view; calendar if you expect stable/
rising IV.** Tie-breaker: the calendar needs mark-to-model management (manage at front expiry).

## Setup 5 — Coiled for a BIG move, LOW IV, 45 days (LOWVOL, spot 185)

IV rank ~15, direction unknown, you expect a breakout. Neutral-but-expecting-a-move + low IV →
**long vol**: long straddle/strangle, or a backspread. Long vol is far easier to justify when IV is
cheap.

In [ ]:
straddle = strategies.long_straddle((185, 4.4), (185, 3.49), expiry=45/365)
strangle_l = strategies.long_strangle((180, 1.69), (190, 2.22), expiry=45/365)
backspr = strategies.call_backspread((185, 4.4), (195, 0.95), expiry=45/365, ratio=(1, 2))
for p in (straddle, strangle_l, backspr):
    brief(p, 185, 0.15)

In [ ]:
ax = viz.plot_compare([straddle, strangle_l, backspr])
ax.set_title("Setup 5: long straddle vs long strangle vs call backspread (LOWVOL)")

**Reasoning.** The **long straddle** profits from a big move either way but costs the most
and bleeds theta if LOWVOL sits still. The **long strangle** is cheaper (wider breakevens, needs a
bigger move). The **call backspread** is a directional long-vol bet (up), often near even money, with
a defined valley — pick it only if you lean up. **Pick: long strangle** for a cheap two-sided
breakout bet in low IV; backspread if you have a directional lean. All three want IV to *rise* —
correct in a low-IV regime.

## Experiments

1. Setup 1: move the condor shorts to ~10-delta (wider: 47.5/52.5/72.5/75). How do credit and POP
   trade off? Re-run `brief`.
2. Setup 2: swap the bull call spread for a bull *put* credit spread and compare — why is the debit
   version the right call in **low** IV?
3. Setup 3: fix the jade lizard so credit >= call-spread width (narrow to 65/67.5). Confirm upside
   risk is gone.
4. Setup 5: raise the flat `vol` you pass to `brief`/`summarize` and watch POP change — long-vol
   trades want vol to rise after entry.
5. Take any setup and change the horizon (DTE). How does the shorter/longer expiry change the
   candidate you would pick?